# Benchmark local de checkpoints

Este notebook compara checkpoints salvos localmente usando o conjunto de teste. Ele mede qualidade preditiva e eficiencia de inferencia:

- loss
- accuracy
- macro-F1
- latencia por imagem
- throughput
- pico de memoria GPU, quando disponivel
- tokens finais e reducao de tokens para modelos podados

A comparacao fica mais justa quando todos os modelos sao medidos na mesma maquina, mesma sessao e mesmo batch size.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repo:", REPO_ROOT)
print("Src:", SRC_DIR)

Repo: c:\Users\ccost\Documents\vit-token-pruning
Src: c:\Users\ccost\Documents\vit-token-pruning\src


In [4]:
import copy
import json
import platform
import time

import pandas as pd
import torch
import torch.nn as nn
import timm

from data.dataset import get_test_dataset, get_test_dataloader
from metrics import calculate_metrics
from model.vit import create_vit_model
from model.pruned_vit import create_pruned_vit_model
from utils import load_config, load_checkpoint

## Configuracao

Ajuste os caminhos abaixo conforme onde os `.pth` estao salvos na sua maquina. O checkpoint do Hybrid pode ficar comentado ate voce treinar/salvar esse modelo.

In [5]:
DATA_DIR = REPO_ROOT / "dataset"
BATCH_SIZE = 32
NUM_WORKERS = 2
WARMUP_BATCHES = 5
MAX_BATCHES = None  # use um inteiro, ex. 10, para teste rapido

BENCHMARKS = [
    {
        "label": "teacher_optuna_best",
        "model_type": "teacher",
        "config_path": REPO_ROOT / "configs" / "optuna_best.yaml",
        "checkpoint_path": Path(r"C:\Users\ccost\Documents\deep-learning\optuna_best_teacher_full\checkpoints\optuna_best_best.pth"),
    },
    {
        "label": "agressive_pruning_topk",
        "model_type": "pruned",
        "config_path": REPO_ROOT / "configs" / "pruning" / "agressive_pruning_topk.yaml",
        "checkpoint_path": Path(r"C:\Users\ccost\Documents\deep-learning\agressive_pruning_topk_results\checkpoints\agressive_pruning_topk_best.pth"),
    },
    {
        "label": "hybrid_history_pruning",
        "model_type": "pruned",
        "config_path": REPO_ROOT / "configs" / "pruning" / "hybrid_history_pruning.yaml",
        "checkpoint_path": Path(r"C:\Users\ccost\Documents\deep-learning\hybrid_history_pruning_results\checkpoints\hybrid_history_pruning_best.pth"),
    },
]

for item in BENCHMARKS:
    print(item["label"])
    print("  config:", item["config_path"], "OK" if item["config_path"].exists() else "NAO ENCONTRADO")
    print("  checkpoint:", item["checkpoint_path"], "OK" if item["checkpoint_path"].exists() else "NAO ENCONTRADO")

teacher_optuna_best
  config: c:\Users\ccost\Documents\vit-token-pruning\configs\optuna_best.yaml OK
  checkpoint: C:\Users\ccost\Documents\deep-learning\optuna_best_teacher_full\checkpoints\optuna_best_best.pth OK
agressive_pruning_topk
  config: c:\Users\ccost\Documents\vit-token-pruning\configs\pruning\agressive_pruning_topk.yaml OK
  checkpoint: C:\Users\ccost\Documents\deep-learning\agressive_pruning_topk_results\checkpoints\agressive_pruning_topk_best.pth OK
hybrid_history_pruning
  config: c:\Users\ccost\Documents\vit-token-pruning\configs\pruning\hybrid_history_pruning.yaml OK
  checkpoint: C:\Users\ccost\Documents\deep-learning\hybrid_history_pruning_results\checkpoints\hybrid_history_pruning_best.pth OK


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"

print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(device))
print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("timm:", timm.__version__)

Device: cpu
Python: 3.11.15
Torch: 2.12.0+cpu
timm: 1.0.27


## Dataset de teste

In [7]:
base_config = load_config(BENCHMARKS[0]["config_path"])
test_dataset = get_test_dataset(
    data_dir=DATA_DIR,
    image_size=base_config["dataset"]["image_size"],
    download=base_config["dataset"].get("download", True),
)
test_loader = get_test_dataloader(
    test_dataset=test_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

print("Tamanho do teste:", len(test_dataset))

Tamanho do teste: 6149


## Funcoes auxiliares

In [8]:
def synchronize_if_needed():
    if device.type == "cuda":
        torch.cuda.synchronize()


def create_model_from_config(config, model_type):
    dataset_config = config["dataset"]
    model_config = config["model"]

    if model_type == "teacher":
        return create_vit_model(
            num_classes=dataset_config["num_classes"],
            pretrained=False,
            model_name=model_config["name"],
        )

    pruning_config = config["pruning"]
    return create_pruned_vit_model(
        num_classes=dataset_config["num_classes"],
        pretrained=False,
        model_name=model_config["name"],
        prune_layers=pruning_config["prune_layers"],
        keep_ratios=pruning_config["keep_ratios"],
        score_method=pruning_config.get("score_method", "token_norm"),
        pruning_method=pruning_config.get("method", "topk"),
        history_config=pruning_config.get("history"),
        preserve_order=pruning_config.get("preserve_order", True),
    )


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())


def model_size_mb(model):
    total_bytes = 0
    for tensor in model.state_dict().values():
        total_bytes += tensor.numel() * tensor.element_size()
    return total_bytes / (1024 ** 2)


def teacher_token_count(model):
    patch_embed = getattr(model, "patch_embed", None)
    if patch_embed is None or not hasattr(patch_embed, "num_patches"):
        return None
    return int(patch_embed.num_patches + 1)

In [9]:
def benchmark_checkpoint(item):
    config = copy.deepcopy(load_config(item["config_path"]))
    config["dataset"]["batch_size"] = BATCH_SIZE
    config["dataset"]["num_workers"] = NUM_WORKERS

    model = create_model_from_config(config, item["model_type"])
    model, checkpoint = load_checkpoint(
        model=model,
        checkpoint_path=item["checkpoint_path"],
        device=device,
    )
    model = model.to(device)
    model.eval()

    criterion = nn.CrossEntropyLoss(
        label_smoothing=config["training"].get("label_smoothing", 0.0)
    )

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    running_loss = 0.0
    evaluated_images = 0
    evaluated_batches = 0
    measured_images = 0
    measured_batches = 0
    total_time = 0.0
    all_outputs = []
    all_targets = []

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(test_loader):
            images = images.to(device, non_blocking=pin_memory)
            labels = labels.to(device, non_blocking=pin_memory)

            synchronize_if_needed()
            start = time.perf_counter()
            outputs = model(images)
            synchronize_if_needed()
            elapsed = time.perf_counter() - start

            loss = criterion(outputs, labels)
            batch_size = images.size(0)

            running_loss += loss.item() * batch_size
            evaluated_images += batch_size
            evaluated_batches += 1
            all_outputs.append(outputs.detach().cpu())
            all_targets.append(labels.detach().cpu())

            if batch_idx >= WARMUP_BATCHES:
                total_time += elapsed
                measured_images += batch_size
                measured_batches += 1

            if MAX_BATCHES is not None and measured_batches >= MAX_BATCHES:
                break

    all_outputs = torch.cat(all_outputs, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    pred_metrics = calculate_metrics(all_outputs, all_targets)

    token_counts = None
    final_tokens = None
    token_reduction = None
    if item["model_type"] == "teacher":
        final_tokens = teacher_token_count(model)
        token_counts = [final_tokens] if final_tokens is not None else None
    elif hasattr(model, "last_token_counts") and model.last_token_counts:
        token_counts = [int(value) for value in model.last_token_counts]
        final_tokens = token_counts[-1]
        full_tokens = teacher_token_count(model.backbone)
        if full_tokens:
            token_reduction = 1 - final_tokens / full_tokens

    peak_memory_mb = None
    if device.type == "cuda":
        peak_memory_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)

    result = {
        "label": item["label"],
        "model_type": item["model_type"],
        "checkpoint_epoch": int(checkpoint.get("epoch", -1)),
        "best_validation_metric": float(checkpoint.get("best_metric", float("nan"))),
        "loss": running_loss / evaluated_images,
        "accuracy": float(pred_metrics["accuracy"]),
        "macro_f1": float(pred_metrics["macro_f1"]),
        "latency_ms_per_image": (total_time / measured_images) * 1000,
        "latency_ms_per_batch": (total_time / measured_batches) * 1000,
        "throughput_images_per_second": measured_images / total_time,
        "peak_memory_mb": peak_memory_mb,
        "num_parameters": int(count_parameters(model)),
        "model_size_mb": float(model_size_mb(model)),
        "token_counts": token_counts,
        "final_tokens": final_tokens,
        "token_reduction_pct": None if token_reduction is None else token_reduction * 100,
        "config_path": str(item["config_path"]),
        "checkpoint_path": str(item["checkpoint_path"]),
    }

    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return result

## Rodar benchmark

In [ ]:
results = []

for item in BENCHMARKS:
    print(f"Rodando: {item['label']}")
    assert item["config_path"].exists(), f"Config nao encontrada: {item['config_path']}"
    assert item["checkpoint_path"].exists(), f"Checkpoint nao encontrado: {item['checkpoint_path']}"
    result = benchmark_checkpoint(item)
    results.append(result)
    print(
        f"  acc={result['accuracy']:.4f} | "
        f"f1={result['macro_f1']:.4f} | "
        f"lat={result['latency_ms_per_image']:.4f} ms/img | "
        f"throughput={result['throughput_images_per_second']:.2f} img/s | "
        f"tokens={result['final_tokens']}"
    )

df = pd.DataFrame(results)
df

Rodando: teacher_optuna_best


KeyboardInterrupt: 

: 

## Comparacao relativa ao teacher

In [ ]:
summary = df.copy()
teacher = summary[summary["model_type"] == "teacher"].iloc[0]

summary["accuracy_delta_pp"] = (summary["accuracy"] - teacher["accuracy"]) * 100
summary["macro_f1_delta_pp"] = (summary["macro_f1"] - teacher["macro_f1"]) * 100
summary["latency_reduction_pct"] = (1 - summary["latency_ms_per_image"] / teacher["latency_ms_per_image"]) * 100
summary["throughput_gain_pct"] = (summary["throughput_images_per_second"] / teacher["throughput_images_per_second"] - 1) * 100
summary["memory_reduction_pct"] = (1 - summary["peak_memory_mb"] / teacher["peak_memory_mb"]) * 100

columns = [
    "label",
    "accuracy",
    "accuracy_delta_pp",
    "macro_f1",
    "macro_f1_delta_pp",
    "final_tokens",
    "token_reduction_pct",
    "latency_ms_per_image",
    "latency_reduction_pct",
    "throughput_images_per_second",
    "throughput_gain_pct",
    "peak_memory_mb",
    "memory_reduction_pct",
]

summary[columns]

## Salvar resultados

In [ ]:
output_dir = REPO_ROOT / "results" / "local_benchmark"
output_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(output_dir / "benchmark_raw.csv", index=False)
summary.to_csv(output_dir / "benchmark_summary.csv", index=False)

with open(output_dir / "benchmark_raw.json", "w", encoding="utf-8") as file:
    json.dump(results, file, indent=4)

print("Resultados salvos em:", output_dir)
print("-", output_dir / "benchmark_raw.csv")
print("-", output_dir / "benchmark_summary.csv")
print("-", output_dir / "benchmark_raw.json")